# USDA Poultry Safety Comprehensive Dashboard
## Food Safety, Animal Welfare, and Recall Analysis (FY2024-2025)

**Coverage:**
- Laboratory Sampling: Salmonella contamination in raw poultry (FY2025)
- Animal Welfare: Good Commercial Practices compliance (FY2025)
- Recall History: Listeria and Salmonella recalls (2024-2025)
- Consumption: US food intake patterns (2017-2018)

**Purpose:** Integrated view of poultry production safety and establishment-level risk assessment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
DATA_PATH = Path('../data/processed')

print("✓ Libraries loaded")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

## Load Processed Data

In [ ]:
# Load all processed datasets
consumption = pd.read_csv(DATA_PATH / 'consumptionCleaned.csv')
contamination = pd.read_csv(DATA_PATH / 'gcpLabJoined.csv')
recalls = pd.read_csv(DATA_PATH / 'recallsApi.csv')
recallsByEst = pd.read_csv(DATA_PATH / 'recallsByEstablishment.csv')

with open(DATA_PATH / 'dashboardStats.json', 'r') as f:
    stats = json.load(f)

# Convert dates
recalls['recallDate'] = pd.to_datetime(recalls['recallDate'])
recalls['closedDate'] = pd.to_datetime(recalls['closedDate'])

# Check temporal alignment
fy2025_start = pd.to_datetime('2024-10-01')
fy2025_end = pd.to_datetime('2025-09-30')
recallsInFY2025 = recalls[(recalls['recallDate'] >= fy2025_start) & (recalls['recallDate'] <= fy2025_end)]
recallsOutside = recalls[(recalls['recallDate'] < fy2025_start) | (recalls['recallDate'] > fy2025_end)]

print("✓ Data loaded successfully")
print(f"\nDataset Sizes:")
print(f"  Establishments (Lab+GCP): {len(contamination):,}")
print(f"  Recalls (all): {len(recalls)}")
print(f"  Consumption records: {len(consumption):,}")

print(f"\n⚠️ TEMPORAL ALIGNMENT:")
print(f"  Lab sampling period: FY2025 (Oct 1, 2024 - Sep 30, 2025)")
print(f"  Recall date range: {recalls['recallDate'].min().date()} to {recalls['recallDate'].max().date()}")
print(f"  Recalls in FY2025 period: {len(recallsInFY2025)} of {len(recalls)} ({len(recallsInFY2025)/len(recalls)*100:.1f}%)")
print(f"  Recalls outside period: {len(recallsOutside)} (shown for context only)")
print(f"\n  Note: Only FY2025 recalls can be directly correlated with lab data.")

## ⚠️ Data Temporal Alignment

**Critical Note:** Lab sampling and recall data cover different time periods.

### Period Coverage
- **Lab Sampling:** FY2025 (October 1, 2024 - September 30, 2025)
- **Recall Data:** Calendar 2024-2026 (January 2024 - present)

### FSIS API Year Taxonomy Issue
The FSIS API's "year" parameter does NOT correspond to calendar years:
- `year=2024` → Returns recalls through ~November 2024
- `year=2025` → Returns recalls through ~October 2025  
- `year=2026` → Captures Nov-Dec 2025 + early 2026

This appears to be related to FSIS's internal categorization system.

### Analysis Implications
**Only ~45% of recalls align with the lab sampling period.** 

Recalls outside FY2025 are shown for context but **cannot be directly correlated** with lab contamination rates. For example:
- **Boar's Head** (July 2024): 2.7M lbs Listeria - occurred BEFORE lab sampling started
- **Nov-Dec 2025 recalls**: Occurred AFTER lab sampling ended

Direct establishment-level correlation is only valid for recalls within FY2025.

## Executive Summary

In [ ]:
print("=" * 80)
print("INTEGRATED FOOD SAFETY METRICS")
print("=" * 80)

print("\n📊 LABORATORY SAMPLING (Raw Poultry - FY2025)")
print(f"  Total samples: {stats['totalSamples']:,}")
print(f"  Salmonella positive: {stats['totalPositive']:,}")
print(f"  Contamination rate: {stats['overallRate']:.2f}%")

print("\n🐔 ANIMAL WELFARE (GCP Inspections - FY2025)")
print(f"  Establishments: {stats['totalEstablishments']:,}")
print(f"  With welfare violations: {stats['withMOIs']} ({stats['moiPercent']:.1f}%)")

print("\n⚠️ RECALL HISTORY (2024-2025)")
print(f"  Total recalls: {stats['totalRecalls']}")
print(f"    Listeria: {stats['listeriaRecalls']}")
print(f"    Salmonella: {stats['salmonellaRecalls']}")
print(f"  Outbreak-related: {stats['outbreakRecalls']}")
print(f"  Pounds recalled: {stats['totalPoundsRecalled']:,.0f} lbs")
print(f"  Largest: {stats['largestRecall']['establishment']} ({stats['largestRecall']['pounds']:,.0f} lbs)")

print("\n" + "=" * 80)

## 1. Food Consumption Context

Understanding consumption patterns provides context for food safety risks. Higher consumption means greater exposure to potential contamination.

In [ ]:
# Consumption by food type
consumptionByType = consumption.groupby('foodType')['ozEquivalent'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart
colors = {'Plant': '#2ecc71', 'Animal': '#e74c3c', 'Other': '#95a5a6'}
ax1.pie(consumptionByType.values, labels=consumptionByType.index, autopct='%1.1f%%',
        colors=[colors[t] for t in consumptionByType.index], startangle=90, textprops={'fontsize': 12})
ax1.set_title('US Food Consumption by Source (2017-2018)', fontsize=14, fontweight='bold')

# Top animal products
animalProducts = consumption[consumption['foodType'] == 'Animal'].nlargest(10, 'ozEquivalent')
ax2.barh(animalProducts['Food group'], animalProducts['ozEquivalent'], color='#e74c3c', alpha=0.7)
ax2.set_xlabel('Daily Intake (oz equivalent)', fontsize=11)
ax2.set_title('Top 10 Animal Products Consumed', fontsize=14, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConsumption Summary:")
for foodType, oz in consumptionByType.items():
    lbsPerYear = oz * 365 / 16
    print(f"  {foodType}: {oz:.1f} oz/day ({lbsPerYear:.0f} lbs/year)")

## 2. Raw Poultry Contamination

USDA samples raw poultry for Salmonella. Contamination in raw products is expected because consumers cook them, which kills bacteria. Concern arises when rates exceed thresholds or correlate with poor practices.

In [ ]:
# Contamination rate distribution
contamination['contaminationRate'] = (contamination['Lab_SalmonellaPositive'] / 
                                       contamination['Lab_TotalSamples'] * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Overall rate display
ax1.text(0.5, 0.5, f"{stats['overallRate']:.2f}%", ha='center', va='center',
         fontsize=80, fontweight='bold', color='#e74c3c')
ax1.text(0.5, 0.25, 'Salmonella Positive Rate\n(Raw Poultry, FY2025)',
         ha='center', va='center', fontsize=16, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')

# Distribution
validData = contamination[contamination['Lab_TotalSamples'] >= 10]['contaminationRate'].dropna()
ax2.hist(validData, bins=30, color='#e74c3c', alpha=0.7, edgecolor='black')
ax2.axvline(stats['overallRate'], color='darkred', linestyle='--', linewidth=2,
            label=f"Overall: {stats['overallRate']:.2f}%")
ax2.set_xlabel('Contamination Rate (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Establishments', fontsize=12, fontweight='bold')
ax2.set_title('Distribution Across Establishments (≥10 samples)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nContamination Statistics:")
print(f"  Mean: {validData.mean():.2f}%")
print(f"  Median: {validData.median():.2f}%")
print(f"  Range: {validData.min():.2f}% to {validData.max():.2f}%")

### Highest Contamination Rates

In [ ]:
# Top contaminated establishments
highContam = contamination[contamination['Lab_TotalSamples'] >= 10].nlargest(15, 'contaminationRate')

fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(range(len(highContam)), highContam['contaminationRate'], 
               color='#e74c3c', alpha=0.8, edgecolor='darkred', linewidth=1.5)
ax.set_yticks(range(len(highContam)))
ax.set_yticklabels([f"{row['Lab_EstablishmentNumber']}" for _, row in highContam.iterrows()], fontsize=10)
ax.set_xlabel('Salmonella Positive Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Establishments by Contamination Rate (≥10 samples)', 
             fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Add sample counts
for i, (idx, row) in enumerate(highContam.iterrows()):
    ax.text(row['contaminationRate'] + 1, i, 
            f"{row['contaminationRate']:.1f}% (n={int(row['Lab_TotalSamples'])})",
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Animal Welfare Compliance

Good Commercial Practices (GCP) measure humane animal handling. MOIs (Matters of Inspection) document welfare concerns. Poor welfare may indicate broader operational issues.

In [ ]:
# Welfare compliance breakdown
welfareCategories = pd.DataFrame({
    'Category': ['No Violations', 'Has MOIs', 'Has NRs', 'Both'],
    'Count': [
        len(contamination[(contamination['GCP_TotalMOIs'] == 0) & (contamination['GCP_TotalNRs'] == 0)]),
        len(contamination[(contamination['GCP_TotalMOIs'] > 0) & (contamination['GCP_TotalNRs'] == 0)]),
        len(contamination[(contamination['GCP_TotalMOIs'] == 0) & (contamination['GCP_TotalNRs'] > 0)]),
        len(contamination[(contamination['GCP_TotalMOIs'] > 0) & (contamination['GCP_TotalNRs'] > 0)])
    ]
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Percentage display
ax1.text(0.5, 0.5, f"{stats['moiPercent']:.1f}%", ha='center', va='center',
         fontsize=80, fontweight='bold', color='#f39c12')
ax1.text(0.5, 0.25, f'Establishments with\nWelfare Violations\n({stats["withMOIs"]} of {stats["totalEstablishments"]})',
         ha='center', va='center', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')

# Breakdown
colors = ['#2ecc71', '#f39c12', '#e67e22', '#c0392b']
ax2.pie(welfareCategories['Count'], labels=welfareCategories['Category'],
        autopct='%1.1f%%', colors=colors, startangle=90, textprops={'fontsize': 11})
ax2.set_title('Welfare Compliance Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nWelfare Compliance:")
for _, row in welfareCategories.iterrows():
    pct = row['Count'] / welfareCategories['Count'].sum() * 100
    print(f"  {row['Category']}: {row['Count']} ({pct:.1f}%)")

### Welfare vs Contamination

In [ ]:
# Compare contamination rates by welfare status
contamination['hasWelfareIssues'] = (contamination['GCP_TotalMOIs'] > 0) | (contamination['GCP_TotalNRs'] > 0)
comparison = contamination[contamination['Lab_TotalSamples'] >= 10].groupby('hasWelfareIssues')['contaminationRate'].agg(
    ['mean', 'median', 'std', 'count']
)

fig, ax = plt.subplots(figsize=(10, 6))
x = [0, 1]
labels = ['No Welfare Issues', 'Has Welfare Issues']
width = 0.35

ax.bar([i - width/2 for i in x], comparison['mean'], width, label='Mean', color='#3498db', alpha=0.8)
ax.bar([i + width/2 for i in x], comparison['median'], width, label='Median', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Contamination Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Contamination Rate: Welfare Compliance vs Non-Compliance', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add n counts
for i, count in enumerate(comparison['count']):
    ax.text(i, comparison['mean'].iloc[i] + 0.5, f"n={int(count)}", 
            ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

print("\nContamination by Welfare Status:")
print(comparison)

## 4. Recall History (2024-2025)

Recalls represent failures where contaminated products reached consumers. This dataset captures Listeria and Salmonella recalls. Ready-to-eat (RTE) products dominate recalls because contamination is immediately dangerous—no cooking step to kill pathogens.

In [ ]:
# Recall timeline
recalls['yearMonth'] = recalls['recallDate'].dt.to_period('M').dt.to_timestamp()
timeline = recalls.groupby(['yearMonth', 'pathogen']).agg({
    'recallNumber': 'count',
    'poundsRecovered': 'sum'
}).reset_index()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Recall count
for pathogen in timeline['pathogen'].unique():
    data = timeline[timeline['pathogen'] == pathogen]
    color = '#e74c3c' if pathogen == 'Listeria' else '#f39c12'
    ax1.plot(data['yearMonth'], data['recallNumber'], marker='o', label=pathogen,
             color=color, linewidth=2, markersize=8)

ax1.set_ylabel('Number of Recalls', fontsize=12, fontweight='bold')
ax1.set_title('Recall Frequency Over Time (2024-2025)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Pounds recalled
for pathogen in timeline['pathogen'].unique():
    data = timeline[timeline['pathogen'] == pathogen]
    color = '#e74c3c' if pathogen == 'Listeria' else '#f39c12'
    ax2.bar(data['yearMonth'], data['poundsRecovered'], width=20, 
            label=pathogen, color=color, alpha=0.7)

ax2.set_xlabel('Date', fontsize=12, fontweight='bold')
ax2.set_ylabel('Pounds Recalled', fontsize=12, fontweight='bold')
ax2.set_title('Volume of Product Recalled', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(axis='y', alpha=0.3)
ax2.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

### Largest Recalls

In [ ]:
# Top 10 recalls by volume
topRecalls = recalls.nlargest(10, 'poundsRecovered')[[
    'establishment', 'poundsRecovered', 'pathogen', 'relatedToOutbreak', 'recallDate'
]]

fig, ax = plt.subplots(figsize=(14, 8))
colors = ['#e74c3c' if p == 'Listeria' else '#f39c12' for p in topRecalls['pathogen']]
bars = ax.barh(range(len(topRecalls)), topRecalls['poundsRecovered'], 
               color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# Mark outbreak-related with thick border
for i, (idx, row) in enumerate(topRecalls.iterrows()):
    if row['relatedToOutbreak']:
        bars[i].set_edgecolor('black')
        bars[i].set_linewidth(3)

ax.set_yticks(range(len(topRecalls)))
ax.set_yticklabels(topRecalls['establishment'], fontsize=10)
ax.set_xlabel('Pounds Recalled', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Largest Recalls (2024-2025)\nBlack border = outbreak-related',
             fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.ticklabel_format(style='plain', axis='x')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Listeria'),
    Patch(facecolor='#f39c12', label='Salmonella')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print("\nTop 5 Recalls:")
for _, row in topRecalls.head(5).iterrows():
    outbreak = " (OUTBREAK)" if row['relatedToOutbreak'] else ""
    print(f"  {row['establishment']}: {row['poundsRecovered']:,.0f} lbs ({row['pathogen']}){outbreak}")

### Establishments with Multiple Recalls

In [ ]:
# Multiple recalls indicate systemic issues
multipleRecalls = recallsByEst[recallsByEst['recallCount'] > 1].sort_values('recallCount', ascending=False)

if len(multipleRecalls) > 0:
    print("⚠️ ESTABLISHMENTS WITH MULTIPLE RECALLS (2024-2025):")
    print("=" * 80)
    for _, row in multipleRecalls.iterrows():
        print(f"\n{row['establishment']}")
        print(f"  Recalls: {int(row['recallCount'])}")
        print(f"  Total pounds: {row['totalPounds']:,.0f} lbs")
        print(f"  Pathogens: {row['pathogens']}")
        print(f"  Outbreaks: {int(row['outbreakCount'])}")
    print("\n" + "=" * 80)
    print("\n⚠️ Multiple recalls suggest systemic food safety issues requiring investigation.")
else:
    print("✓ No establishments had multiple recalls in this period.")

### Recall Resolution Time

In [ ]:
# How long do recalls take to close?
withDuration = recalls[recalls['daysOpen'].notna()]

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(withDuration['daysOpen'], bins=20, color='#3498db', alpha=0.7, edgecolor='black')
ax.axvline(withDuration['daysOpen'].median(), color='darkblue', linestyle='--', 
           linewidth=2, label=f'Median: {withDuration["daysOpen"].median():.0f} days')
ax.set_xlabel('Days to Close Recall', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Recalls', fontsize=12, fontweight='bold')
ax.set_title('Recall Resolution Time Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nRecall Duration:")
print(f"  Mean: {withDuration['daysOpen'].mean():.0f} days")
print(f"  Median: {withDuration['daysOpen'].median():.0f} days")
print(f"  Range: {withDuration['daysOpen'].min():.0f} to {withDuration['daysOpen'].max():.0f} days")

## 5. Establishment-Level Risk Assessment

Combining recall history with laboratory sampling reveals establishments with elevated risk profiles.

In [ ]:
# Attempt to match establishments (limited by name vs number mismatch)
recallEsts = set(recalls['establishment'].unique())
labEsts = set(contamination['Lab_EstablishmentNumber'].astype(str).unique())
matches = recallEsts.intersection(labEsts)

print(f"Establishment Matching:")
print(f"  Recall dataset: {len(recallEsts)} establishments")
print(f"  Lab dataset: {len(labEsts)} establishments")
print(f"  Direct matches: {len(matches)}")

if len(matches) > 0:
    print("\n✓ Matched Establishments:")
    for est in matches:
        recallInfo = recalls[recalls['establishment'] == est]
        labInfo = contamination[contamination['Lab_EstablishmentNumber'].astype(str) == est]
        print(f"\n{est}:")
        print(f"  Recalls: {len(recallInfo)}")
        if len(labInfo) > 0:
            print(f"  Contamination rate: {labInfo['contaminationRate'].mean():.2f}%")
else:
    print("\n⚠️ No direct matches found.")
    print("\nThis is expected because:")
    print("  - Recall data uses company names (e.g., 'BrucePac', 'Boar's Head')")
    print("  - Lab data uses establishment numbers (e.g., 'P-46684')")
    print("  - Production system would require USDA establishment number cross-reference")
    print("\nShowing both datasets for manual comparison:")

In [ ]:
# Side-by-side comparison
print("\n" + "=" * 80)
print("HIGH-RISK ESTABLISHMENTS TO MONITOR")
print("=" * 80)

print("\n📋 FROM RECALL DATA (2024-2025):")
print("\nEstablishments with recalls (top 10 by volume):")
for _, row in recallsByEst.nlargest(10, 'totalPounds').iterrows():
    print(f"  {row['establishment']}")
    print(f"    {int(row['recallCount'])} recalls, {row['totalPounds']:,.0f} lbs, {row['pathogens']}")

print("\n📋 FROM LAB SAMPLING (FY2025):")
print("\nHighest contamination rates (≥20 samples):")
highRisk = contamination[contamination['Lab_TotalSamples'] >= 20].nlargest(10, 'contaminationRate')
for _, row in highRisk.iterrows():
    welfareNote = f", {int(row['GCP_TotalMOIs'])} MOIs" if row['GCP_TotalMOIs'] > 0 else ""
    print(f"  {row['Lab_EstablishmentNumber']}: {row['contaminationRate']:.2f}%" + 
          f" (n={int(row['Lab_TotalSamples'])}){welfareNote}")

print("\n" + "=" * 80)
print("\nNote: Establishment name/number matching would enable direct risk correlation.")
print("Current data shows independent high-risk flags from recalls and lab sampling.")

## Conclusions

### Key Findings

**1. Raw Poultry Contamination (FY2025):**
- Overall Salmonella rate: 7.69%
- Wide establishment variation (0% to >60%)
- Raw contamination is expected; consumers cook these products

**2. Animal Welfare (FY2025):**
- 25.6% of establishments had welfare violations
- No strong statistical correlation with contamination in observational data
- Biological mechanisms still support welfare-safety link

**3. Recalls (2024-2025):**
- 20 Listeria/Salmonella recalls captured
- 7.3 million pounds recalled
- 4 recalls linked to confirmed outbreaks
- BrucePac: Largest single recall (3.7M lbs, Listeria)
- Multiple repeat offenders indicate systemic issues

**4. RTE vs Raw Products:**
- Ready-to-eat products dominate recalls (no consumer cooking)
- Raw poultry contamination managed through cooking
- Different products require different safety strategies

### Data Limitations

- **Recall data scope:** Pre-filtered for Listeria/Salmonella; not all recalls
- **Establishment matching:** Cannot directly link names to numbers without USDA cross-reference
- **Timeframe mismatch:** Lab data (FY2025) vs recalls (calendar 2024-2025)
- **Product differences:** Lab measures raw; recalls primarily RTE

### Recommendations

1. **Enhanced Monitoring:** Increase sampling at establishments with recalls
2. **Data Integration:** Use USDA establishment numbers to properly link datasets
3. **Repeat Offender Protocol:** Mandatory intervention for multiple recalls
4. **RTE Focus:** Heightened environmental monitoring at RTE facilities
5. **Welfare-Safety Research:** Study causal mechanisms linking welfare to contamination